In [729]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [730]:
# fetch data 

mco_syr_sw_data = pd.read_csv('flight_data\mco_syr_sw_22_23.csv')
mco_syr_jb_data = pd.read_csv('flight_data\mco_syr_jb_22_23.csv')
jfk_syr_jb_data = pd.read_csv('flight_data\jfk_syr_jb_22_23.csv')
jfk_syr_end_data = pd.read_csv('flight_data\jfk_syr_end_22_23.csv')
ord_syr_ua_data = pd.read_csv('flight_data\ord_syr_ua_22_23.csv')
ord_syr_aa_data = pd.read_csv('flight_data\ord_syr_aa_22_23.csv')

ord_syr_ua_data['dep_order'] = 'early'
jfk_syr_jb_data['dep_order'] = 'early'
mco_syr_sw_data['dep_order'] = 'early'
jfk_syr_end_data['dep_order'] = 'latter'
ord_syr_aa_data['dep_order'] = 'latter'
mco_syr_jb_data['dep_order'] = 'latter'
#['dep_order'] = 'early'
# mco_syr_sw_data.head()
# mco_syr_jb_data.head()
# jfk_syr_jb_data.head()
# jfk_syr_end_data.head()
# ord_syr_ua_data.head()
# ord_syr_aa_data.head()

In [731]:
dfs = [ord_syr_aa_data,ord_syr_ua_data,jfk_syr_end_data,jfk_syr_jb_data,mco_syr_jb_data,mco_syr_sw_data]
main_data = pd.concat(dfs,axis = 0)
main_data.head()
len(main_data)

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,Wheels-on Time,Taxi-In time (Minutes),arr_Delay_Carrier,arr_Delay_Weather,arr_Delay_National_Aviation_System,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year
0,0,MQ,2022-01-01,4316,N900AE,SYR,15:28,0:00,107,0,...,0:00,0,0,0,0,0,0,18,5,2022
1,1,MQ,2023-01-01,3392,N283NN,SYR,18:25,18:24,111,104,...,21:02,6,0,0,0,0,0,21,6,2023
2,2,MQ,2023-01-01,3518,N213NN,SYR,13:05,13:01,117,96,...,15:33,4,0,0,0,0,0,16,6,2023
3,3,MQ,2022-01-02,4134,N900AE,SYR,10:25,10:21,111,146,...,13:43,4,0,0,31,0,0,13,6,2022
4,4,MQ,2022-01-02,4253,N238NN,SYR,17:25,17:51,107,116,...,20:40,7,0,26,9,0,0,20,6,2022


5623

In [732]:
main_data.dtypes
# main_data.dtypes['dep_min']



Unnamed: 0                             int64
Carrier_Code                          object
Date                                  object
Flight_Number                          int64
Tail_Number                           object
Destination_Airport                   object
Scheduled departure time              object
Actual departure time                 object
Scheduled elapsed time (Minutes)       int64
Actual elapsed time (Minutes)          int64
Departure delay (Minutes)              int64
Wheels-off time                       object
Taxi-Out time (Minutes)                int64
dep_Delay_Carrier                      int64
dep_Delay_Weather                      int64
dep_Delay_National_Aviation_System     int64
dep_Delay_Security                     int64
dep_Delay_Late_Aircraft_Arrival        int64
dep_hour                               int64
dep_day                                int64
dep_year                               int64
dep_order                             object
Origin_Air

In [733]:
def round_to_nearest_quarter(minute):
    if 0 <= minute < 14:
        return 0
    elif 15 <= minute < 29:
        return 15
    elif 30 <= minute < 44:
        return 30
    else:
        return 45
    
main_data['dep_min'] = main_data['Scheduled departure time'].str.split(":").str[1].astype('int64')
main_data['dep_minutes'] = main_data['dep_min'].apply(round_to_nearest_quarter).astype('object')
main_data['dep_min'] = main_data['dep_min'].astype('object')
main_data['arr_min'] = main_data['Scheduled Arrival Time'].str.split(":").str[1].astype('int64')
main_data['arr_minutes'] = main_data['arr_min'].apply(round_to_nearest_quarter).astype('object')
main_data['arr_min'] = main_data['arr_min'].astype('object')
main_data['Flight_Number'] = main_data['Flight_Number'].astype('object')
main_data['dep_hours'] = main_data['dep_hour'].astype('object')
main_data['dep_hour'] = main_data['dep_hour'].astype('object')
main_data['dep_day'] = main_data['dep_day'].astype('object')
main_data['arr_hours'] = main_data['arr_hour'].astype('object')
main_data['arr_hour'] = main_data['arr_hour'].astype('object')
main_data['arr_day'] = main_data['arr_day'].astype('object')
conditions = [
    (main_data['Arrival Delay (Minutes)'] > 5),
    (main_data['Arrival Delay (Minutes)'] >= -5) & (main_data['Arrival Delay (Minutes)'] <= 5),
    (main_data['Arrival Delay (Minutes)'] < -5)
]
choices = [2,1,0]
main_data['arr_status'] = np.select(conditions, choices)
main_data.dtypes
main_data.head()

Unnamed: 0                             int64
Carrier_Code                          object
Date                                  object
Flight_Number                         object
Tail_Number                           object
Destination_Airport                   object
Scheduled departure time              object
Actual departure time                 object
Scheduled elapsed time (Minutes)       int64
Actual elapsed time (Minutes)          int64
Departure delay (Minutes)              int64
Wheels-off time                       object
Taxi-Out time (Minutes)                int64
dep_Delay_Carrier                      int64
dep_Delay_Weather                      int64
dep_Delay_National_Aviation_System     int64
dep_Delay_Security                     int64
dep_Delay_Late_Aircraft_Arrival        int64
dep_hour                              object
dep_day                               object
dep_year                               int64
dep_order                             object
Origin_Air

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,arr_hour,arr_day,arr_year,dep_min,dep_minutes,arr_min,arr_minutes,dep_hours,arr_hours,arr_status
0,0,MQ,2022-01-01,4316,N900AE,SYR,15:28,0:00,107,0,...,18,5,2022,28,15,15,15,15,18,1
1,1,MQ,2023-01-01,3392,N283NN,SYR,18:25,18:24,111,104,...,21,6,2023,25,15,16,15,18,21,0
2,2,MQ,2023-01-01,3518,N213NN,SYR,13:05,13:01,117,96,...,16,6,2023,5,0,2,0,13,16,0
3,3,MQ,2022-01-02,4134,N900AE,SYR,10:25,10:21,111,146,...,13,6,2022,25,15,16,15,10,13,2
4,4,MQ,2022-01-02,4253,N238NN,SYR,17:25,17:51,107,116,...,20,6,2022,25,15,12,0,17,20,2


In [734]:
sub_data = main_data.drop(columns = ['Unnamed: 0','Destination_Airport','Actual departure time','Scheduled elapsed time (Minutes)','Actual elapsed time (Minutes)',
                         'Departure delay (Minutes)','Wheels-off time','Taxi-Out time (Minutes)','dep_Delay_Carrier', 'dep_year',
                         'dep_Delay_Weather','dep_Delay_National_Aviation_System','dep_Delay_Security','dep_Delay_Late_Aircraft_Arrival','Actual Arrival Time','Wheels-on Time','Taxi-In time (Minutes)','arr_Delay_Carrier','arr_Delay_Weather',
                         'arr_Delay_National_Aviation_System','arr_Delay_Security','arr_Delay_Late_Aircraft_Arrival','arr_year'])

In [735]:
sub_data.dtypes

Carrier_Code                object
Date                        object
Flight_Number               object
Tail_Number                 object
Scheduled departure time    object
dep_hour                    object
dep_day                     object
dep_order                   object
Origin_Airport              object
Scheduled Arrival Time      object
Arrival Delay (Minutes)      int64
arr_hour                    object
arr_day                     object
dep_min                     object
dep_minutes                 object
arr_min                     object
arr_minutes                 object
dep_hours                   object
arr_hours                   object
arr_status                   int32
dtype: object

In [736]:
sub_data.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day,dep_min,dep_minutes,arr_min,arr_minutes,dep_hours,arr_hours,arr_status
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,0,18,5,28,15,15,15,15,18,1
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,-8,21,6,25,15,16,15,18,21,0
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,-25,16,6,5,0,2,0,13,16,0
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,31,13,6,25,15,16,15,10,13,2
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,35,20,6,25,15,12,0,17,20,2


In [737]:
latter_data = sub_data[sub_data['dep_order'] == 'latter']
len(latter_data)

3349

In [738]:
early_data = sub_data[sub_data['dep_order'] == 'early']
len(early_data)

2274

In [739]:
# latter_data['dep_year'].value_counts()

In [740]:
# early_data['dep_year'].value_counts()

In [741]:
# latter_data['Carrier_Code'].value_counts()

In [742]:
# early_data['Carrier_Code'].value_counts()

In [743]:
merged_df = pd.merge(latter_data, early_data, on=['Date', 'Origin_Airport'], how = 'left')
merged_df.head()
len(merged_df)

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_order_x,Origin_Airport,Scheduled Arrival Time_x,...,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,dep_minutes_y,arr_min_y,arr_minutes_y,dep_hours_y,arr_hours_y,arr_status_y
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,...,-10.00,23,6,10,0,57,45,21,23,0.00
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,...,-10.00,23,6,10,0,57,45,21,23,0.00
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


4991

In [744]:
merged_df.head(20)

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_order_x,Origin_Airport,Scheduled Arrival Time_x,...,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,dep_minutes_y,arr_min_y,arr_minutes_y,dep_hours_y,arr_hours_y,arr_status_y
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,latter,ORD,18:15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,latter,ORD,21:16,...,-10.00,23,6,10,0,57,45,21,23,0.00
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,latter,ORD,16:02,...,-10.00,23,6,10,0,57,45,21,23,0.00
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,latter,ORD,13:16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,latter,ORD,20:12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,MQ,2022-01-02,4316,N922AE,15:28,15,6,latter,ORD,18:15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,MQ,2023-01-02,3392,N248NN,17:33,17,0,latter,ORD,20:19,...,-21.00,21,0,14,45,7,0,18,21,0.00
7,MQ,2023-01-02,3518,N263NN,13:05,13,0,latter,ORD,16:02,...,-21.00,21,0,14,45,7,0,18,21,0.00
8,MQ,2022-01-03,4134,N606AE,10:25,10,0,latter,ORD,13:16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,MQ,2022-01-03,4253,N200NN,17:25,17,0,latter,ORD,20:12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [745]:
merged_df.columns

Index(['Carrier_Code_x', 'Date', 'Flight_Number_x', 'Tail_Number_x',
       'Scheduled departure time_x', 'dep_hour_x', 'dep_day_x', 'dep_order_x',
       'Origin_Airport', 'Scheduled Arrival Time_x',
       'Arrival Delay (Minutes)_x', 'arr_hour_x', 'arr_day_x', 'dep_min_x',
       'dep_minutes_x', 'arr_min_x', 'arr_minutes_x', 'dep_hours_x',
       'arr_hours_x', 'arr_status_x', 'Carrier_Code_y', 'Flight_Number_y',
       'Tail_Number_y', 'Scheduled departure time_y', 'dep_hour_y',
       'dep_day_y', 'dep_order_y', 'Scheduled Arrival Time_y',
       'Arrival Delay (Minutes)_y', 'arr_hour_y', 'arr_day_y', 'dep_min_y',
       'dep_minutes_y', 'arr_min_y', 'arr_minutes_y', 'dep_hours_y',
       'arr_hours_y', 'arr_status_y'],
      dtype='object')

In [746]:
merged_df['hour_diff']= merged_df['dep_hour_x'] - merged_df['dep_hour_y']
merged_df1 = merged_df[merged_df['hour_diff'] ==0 ]
merged_df1 = merged_df1[merged_df1['dep_min_x'] > merged_df1['dep_min_y']]
merged_df2 = merged_df[merged_df['hour_diff'] ==1]
merged_df = pd.concat([merged_df1,merged_df2])
len(merged_df)

685

In [747]:
# merged_df['dep_year_x'].value_counts()

In [748]:
merged_df["Carrier_Code_x"] = np.where(merged_df["Carrier_Code_x"] == "MQ", "AA", merged_df["Carrier_Code_x"])
merged_df["Carrier_Code_x"] = np.where(merged_df["Carrier_Code_x"] == "9E", "DL", merged_df["Carrier_Code_x"])

In [749]:
merged_df.columns

Index(['Carrier_Code_x', 'Date', 'Flight_Number_x', 'Tail_Number_x',
       'Scheduled departure time_x', 'dep_hour_x', 'dep_day_x', 'dep_order_x',
       'Origin_Airport', 'Scheduled Arrival Time_x',
       'Arrival Delay (Minutes)_x', 'arr_hour_x', 'arr_day_x', 'dep_min_x',
       'dep_minutes_x', 'arr_min_x', 'arr_minutes_x', 'dep_hours_x',
       'arr_hours_x', 'arr_status_x', 'Carrier_Code_y', 'Flight_Number_y',
       'Tail_Number_y', 'Scheduled departure time_y', 'dep_hour_y',
       'dep_day_y', 'dep_order_y', 'Scheduled Arrival Time_y',
       'Arrival Delay (Minutes)_y', 'arr_hour_y', 'arr_day_y', 'dep_min_y',
       'dep_minutes_y', 'arr_min_y', 'arr_minutes_y', 'dep_hours_y',
       'arr_hours_y', 'arr_status_y', 'hour_diff'],
      dtype='object')

In [750]:
merged_df.drop(columns=['Carrier_Code_y', 'Flight_Number_y',
       'Tail_Number_y', 'Scheduled departure time_y', 'dep_hour_y',
       'dep_day_y', 'dep_order_y', 'Scheduled Arrival Time_y',
       'Arrival Delay (Minutes)_y', 'arr_hour_y', 'arr_day_y', 'dep_min_y',
       'dep_minutes_y', 'arr_min_y', 'arr_minutes_y', 'dep_hours_y',
       'arr_hours_y'], inplace=True)

In [751]:
merged_df.columns = merged_df.columns.str.replace('_x', '')
merged_df.rename(columns={'arr_status':'arr_status_x'}, inplace=True)

merged_df.head()
merged_df.columns

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,...,arr_day,dep_min,dep_minutes,arr_min,arr_minutes,dep_hours,arr_hours,arr_status_x,arr_status_y,hour_diff
313,AA,2023-07-04,3402,N298FR,18:30,18,1,latter,ORD,21:25,...,1,30,30,25,15,18,21,1,2.00,0
314,AA,2023-07-05,3402,N768RD,18:36,18,2,latter,ORD,21:32,...,2,36,30,32,30,18,21,1,2.00,0
317,AA,2023-07-06,3402,N634RW,18:36,18,3,latter,ORD,21:32,...,3,36,30,32,30,18,21,0,0.00,0
328,AA,2023-07-10,3402,N228NN,18:36,18,0,latter,ORD,21:32,...,0,36,30,32,30,18,21,2,2.00,0
331,AA,2023-07-11,3402,N449YX,18:36,18,1,latter,ORD,21:32,...,1,36,30,32,30,18,21,2,1.00,0


Index(['Carrier_Code', 'Date', 'Flight_Number', 'Tail_Number',
       'Scheduled departure time', 'dep_hour', 'dep_day', 'dep_order',
       'Origin_Airport', 'Scheduled Arrival Time', 'Arrival Delay (Minutes)',
       'arr_hour', 'arr_day', 'dep_min', 'dep_minutes', 'arr_min',
       'arr_minutes', 'dep_hours', 'arr_hours', 'arr_status_x', 'arr_status_y',
       'hour_diff'],
      dtype='object')

In [752]:
# merged_df.to_csv('flight_data\\all_lat_ear.csv')

In [753]:
# Read and process weather data files for each airport
jfk_weather_data = pd.read_csv('weather_data/JFK_weather_data_sub_hourly_processed.csv')
syr_weather_data = pd.read_csv('weather_data/SYR_weather_data_sub_hourly_processed.csv')
ord_weather_data = pd.read_csv('weather_data/ORD_weather_data_sub_hourly_processed.csv')
mco_weather_data = pd.read_csv('weather_data/MCO_weather_data_sub_hourly_processed.csv')

# Combine weather data for all airports
weather_dfs = [jfk_weather_data, ord_weather_data, mco_weather_data]
weather_data = pd.concat(weather_dfs, axis=0)

# Define merging logic based on airport code
merged_df = pd.merge(merged_df, syr_weather_data, how='left', on=['Date', 'arr_hours', 'arr_minutes'])

merged_df = pd.merge(merged_df, weather_data, how='left', on=['Origin_Airport', 'Date', 'dep_hours', 'dep_minutes'])


merged_df.head()


,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,...,dep_pres,dep_rh,dep_snow_rate,dep_temp,dep_vis,dep_weather.description,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,AA,2023-07-04,3402,N298FR,18:30,18,1,latter,ORD,21:25,...,989,52,0.00,30.40,16,Broken clouds,803,290,4.40,2.15
1,AA,2023-07-05,3402,N768RD,18:36,18,2,latter,ORD,21:32,...,987,84,0.00,23.90,9,Heavy rain,502,295,10.30,4.30
2,AA,2023-07-06,3402,N634RW,18:36,18,3,latter,ORD,21:32,...,989,55,0.00,24.20,16,Overcast clouds,804,75,6.20,3.72
3,AA,2023-07-10,3402,N228NN,18:36,18,0,latter,ORD,21:32,...,987,45,0.00,28.60,16,Broken clouds,803,210,9.10,4.50
4,AA,2023-07-11,3402,N449YX,18:36,18,1,latter,ORD,21:32,...,989,65,0.00,22.10,16,Scattered clouds,802,70,6.20,3.10


In [754]:
# merged_df.to_csv('flight_data\\all_lat_ear.csv')

In [755]:
merged_df.drop(columns= ['Date', 'Flight_Number', 'Tail_Number',
       'Scheduled departure time','Carrier_Code',
       'dep_order', 'Scheduled Arrival Time',
       'Arrival Delay (Minutes)','hour_diff','arr_weather.description','dep_weather.description'],inplace = True)

merged_df.dtypes

dep_hour              object
dep_day               object
Origin_Airport        object
arr_hour              object
arr_day               object
dep_min               object
dep_minutes           object
arr_min               object
arr_minutes           object
dep_hours             object
arr_hours             object
arr_status_x           int32
arr_status_y         float64
arr_clouds             int64
arr_dewpt            float64
arr_precip_rate      float64
arr_pres               int64
arr_rh                 int64
arr_snow_rate        float64
arr_temp             float64
arr_vis                int64
arr_weather.code       int64
arr_wind_dir           int64
arr_wind_gust_spd    float64
arr_wind_spd         float64
dep_clouds             int64
dep_dewpt            float64
dep_precip_rate      float64
dep_pres               int64
dep_rh                 int64
dep_snow_rate        float64
dep_temp             float64
dep_vis                int64
dep_weather.code       int64
dep_wind_dir  

In [756]:
merged_df['arr_status_y'] = merged_df['arr_status_y'].astype('int64').astype('object')
merged_df.head()
merged_df.dtypes
su_data = merged_df

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,dep_minutes,arr_min,arr_minutes,dep_hours,...,dep_precip_rate,dep_pres,dep_rh,dep_snow_rate,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,1,ORD,21,1,30,30,25,15,18,...,0.00,989,52,0.00,30.40,16,803,290,4.40,2.15
1,18,2,ORD,21,2,36,30,32,30,18,...,7.00,987,84,0.00,23.90,9,502,295,10.30,4.30
2,18,3,ORD,21,3,36,30,32,30,18,...,0.00,989,55,0.00,24.20,16,804,75,6.20,3.72
3,18,0,ORD,21,0,36,30,32,30,18,...,0.00,987,45,0.00,28.60,16,803,210,9.10,4.50
4,18,1,ORD,21,1,36,30,32,30,18,...,0.00,989,65,0.00,22.10,16,802,70,6.20,3.10


dep_hour              object
dep_day               object
Origin_Airport        object
arr_hour              object
arr_day               object
dep_min               object
dep_minutes           object
arr_min               object
arr_minutes           object
dep_hours             object
arr_hours             object
arr_status_x           int32
arr_status_y          object
arr_clouds             int64
arr_dewpt            float64
arr_precip_rate      float64
arr_pres               int64
arr_rh                 int64
arr_snow_rate        float64
arr_temp             float64
arr_vis                int64
arr_weather.code       int64
arr_wind_dir           int64
arr_wind_gust_spd    float64
arr_wind_spd         float64
dep_clouds             int64
dep_dewpt            float64
dep_precip_rate      float64
dep_pres               int64
dep_rh                 int64
dep_snow_rate        float64
dep_temp             float64
dep_vis                int64
dep_weather.code       int64
dep_wind_dir  

In [757]:
# su_data = sub_data.drop(columns = ['dep_order'])
su_data.head()

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,dep_minutes,arr_min,arr_minutes,dep_hours,...,dep_precip_rate,dep_pres,dep_rh,dep_snow_rate,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,1,ORD,21,1,30,30,25,15,18,...,0.00,989,52,0.00,30.40,16,803,290,4.40,2.15
1,18,2,ORD,21,2,36,30,32,30,18,...,7.00,987,84,0.00,23.90,9,502,295,10.30,4.30
2,18,3,ORD,21,3,36,30,32,30,18,...,0.00,989,55,0.00,24.20,16,804,75,6.20,3.72
3,18,0,ORD,21,0,36,30,32,30,18,...,0.00,987,45,0.00,28.60,16,803,210,9.10,4.50
4,18,1,ORD,21,1,36,30,32,30,18,...,0.00,989,65,0.00,22.10,16,802,70,6.20,3.10


In [758]:
su_data.columns
su_data.head()

Index(['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'dep_minutes', 'arr_min', 'arr_minutes', 'dep_hours',
       'arr_hours', 'arr_status_x', 'arr_status_y', 'arr_clouds', 'arr_dewpt',
       'arr_precip_rate', 'arr_pres', 'arr_rh', 'arr_snow_rate', 'arr_temp',
       'arr_vis', 'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd',
       'arr_wind_spd', 'dep_clouds', 'dep_dewpt', 'dep_precip_rate',
       'dep_pres', 'dep_rh', 'dep_snow_rate', 'dep_temp', 'dep_vis',
       'dep_weather.code', 'dep_wind_dir', 'dep_wind_gust_spd',
       'dep_wind_spd'],
      dtype='object')

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,dep_minutes,arr_min,arr_minutes,dep_hours,...,dep_precip_rate,dep_pres,dep_rh,dep_snow_rate,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,1,ORD,21,1,30,30,25,15,18,...,0.00,989,52,0.00,30.40,16,803,290,4.40,2.15
1,18,2,ORD,21,2,36,30,32,30,18,...,7.00,987,84,0.00,23.90,9,502,295,10.30,4.30
2,18,3,ORD,21,3,36,30,32,30,18,...,0.00,989,55,0.00,24.20,16,804,75,6.20,3.72
3,18,0,ORD,21,0,36,30,32,30,18,...,0.00,987,45,0.00,28.60,16,803,210,9.10,4.50
4,18,1,ORD,21,1,36,30,32,30,18,...,0.00,989,65,0.00,22.10,16,802,70,6.20,3.10


In [759]:

su_data['dep_hour'] = pd.Categorical(su_data['dep_hour'], categories=[i for i in range(24)])
su_data['dep_day'] = pd.Categorical(su_data['dep_day'], categories=[i for i in range(7)])
su_data['dep_min'] = pd.Categorical(su_data['dep_min'], categories=[i for i in range(60)])
su_data['arr_hour'] = pd.Categorical(su_data['arr_hour'], categories=[i for i in range(24)])
su_data['arr_day'] = pd.Categorical(su_data['arr_day'], categories=[i for i in range(7)])
su_data['arr_min'] = pd.Categorical(su_data['arr_min'], categories=[i for i in range(60)])
#su_data['Carrier_Code'] = pd.Categorical(su_data['Carrier_Code'], categories=['AA', 'DL', 'B6'])
su_data['Origin_Airport'] = pd.Categorical(su_data['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])
su_data['arr_weather.code'] = pd.Categorical(su_data['arr_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
su_data['dep_weather.code'] = pd.Categorical(su_data['dep_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
su_data.drop(columns=['arr_precip_rate', 'arr_snow_rate', 'dep_precip_rate', 'dep_snow_rate','dep_hours',
                     'arr_hours','arr_minutes','dep_minutes'],inplace = True)
su_data.columns

Index(['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'arr_status_x', 'arr_status_y', 'arr_clouds',
       'arr_dewpt', 'arr_pres', 'arr_rh', 'arr_temp', 'arr_vis',
       'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd', 'arr_wind_spd',
       'dep_clouds', 'dep_dewpt', 'dep_pres', 'dep_rh', 'dep_temp', 'dep_vis',
       'dep_weather.code', 'dep_wind_dir', 'dep_wind_gust_spd',
       'dep_wind_spd'],
      dtype='object')

In [760]:
su_data = pd.get_dummies(su_data, drop_first = True)
#su_data['arr_status'] = su_data['arr_status'].astype('object')
su_data.head()
su_data.dtypes

,arr_status_x,arr_clouds,arr_dewpt,arr_pres,arr_rh,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
0,1,43,18.20,999,70,24.00,13,320,2.00,1.10,...,False,False,False,False,False,False,False,True,False,False
1,1,25,17.70,999,65,24.80,16,100,1.00,0.75,...,False,False,False,False,False,False,False,False,False,False
2,0,87,18.50,995,62,26.40,16,260,3.60,1.30,...,False,False,False,False,False,False,False,False,True,False
3,2,87,15.50,996,85,18.10,16,180,2.80,1.50,...,False,False,False,False,False,False,False,True,False,False
4,2,68,17.40,996,65,24.40,16,225,3.00,1.50,...,False,False,False,False,False,False,True,False,False,False


arr_status_x              int32
arr_clouds                int64
arr_dewpt               float64
arr_pres                  int64
arr_rh                    int64
                         ...   
dep_weather.code_801       bool
dep_weather.code_802       bool
dep_weather.code_803       bool
dep_weather.code_804       bool
dep_weather.code_900       bool
Length: 269, dtype: object

In [761]:
X_train, X_test, y_train, y_test = train_test_split(su_data.drop(columns = ['arr_status_x']), su_data['arr_status_x'], test_size=0.2, stratify = su_data['arr_status_x'], random_state=50)
# In the above split the stratify = y essentially makes sure the fractions of the classification is maintained
X_train
X_test
y_train.dtypes
y_test

,arr_clouds,arr_dewpt,arr_pres,arr_rh,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd,dep_clouds,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
101,100,3.30,996,38,17.80,16,260,12.80,6.70,0,...,False,False,False,False,True,False,False,False,False,False
243,25,-4.10,1003,41,8.30,16,190,3.07,3.07,100,...,False,False,False,False,False,False,False,False,False,False
110,87,2.80,985,49,13.30,16,230,8.20,3.97,100,...,False,False,False,False,False,False,False,False,True,False
313,100,1.50,989,95,2.20,16,250,7.70,2.43,50,...,False,False,False,False,False,False,False,True,False,False
355,100,-3.80,1002,67,1.70,16,245,5.00,3.22,25,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23,100,17.10,999,87,19.30,9,220,3.35,3.35,50,...,False,False,False,False,False,False,False,True,False,False
638,87,1.10,1002,65,7.20,16,250,8.50,4.59,18,...,False,False,False,False,False,True,False,False,False,False
378,59,-1.70,998,74,2.50,16,280,8.90,4.62,78,...,False,False,False,False,False,False,False,False,True,False
301,87,10.90,1011,73,15.70,16,90,4.90,2.60,87,...,False,False,False,False,False,False,False,False,True,False


,arr_clouds,arr_dewpt,arr_pres,arr_rh,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd,dep_clouds,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
476,31,14.30,1000,74,19.00,16,205,4.10,2.10,100,...,False,False,False,False,False,False,False,False,True,False
418,100,12.40,985,89,14.20,12,285,10.30,5.50,87,...,False,False,False,False,False,False,False,False,True,False
667,100,-1.80,992,95,-1.10,3,290,10.20,4.09,87,...,False,False,False,False,False,False,False,False,True,False
423,50,12.10,1007,62,19.60,16,90,5.20,2.32,87,...,False,False,False,False,False,False,False,False,True,False
549,100,2.60,1008,66,8.60,16,270,9.50,3.84,50,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,87,5.60,995,22,29.50,16,300,3.40,2.47,87,...,False,False,False,False,False,False,False,False,True,False
628,87,-9.50,1004,45,1.10,16,220,9.80,7.20,25,...,False,False,False,False,False,False,True,False,False,False
518,87,16.60,997,65,23.60,16,125,6.90,0.77,25,...,False,False,False,False,False,False,True,False,False,False
172,100,-4.40,1014,59,2.80,16,260,2.10,2.10,25,...,False,False,False,False,False,False,True,False,False,False


dtype('int32')

476    0
418    0
667    0
423    2
549    0
      ..
104    2
628    2
518    1
172    0
2      0
Name: arr_status_x, Length: 137, dtype: int32

In [762]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = pd.DataFrame(sc.fit_transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(sc.transform(X_test), columns = X_test.columns, index = X_test.index)
X_train
X_test
y_train
y_test

,arr_clouds,arr_dewpt,arr_pres,arr_rh,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd,dep_clouds,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
101,0.80,-0.20,-0.54,-1.81,0.74,0.43,0.75,1.83,1.62,-2.08,...,-0.09,0.00,-0.18,0.00,4.67,-0.14,-0.51,-0.40,-0.99,0.00
243,-1.60,-1.07,0.34,-1.63,-0.33,0.43,-0.04,-1.03,-0.12,1.02,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,-0.99,0.00
110,0.39,-0.26,-1.92,-1.15,0.23,0.43,0.41,0.47,0.31,1.02,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00
313,0.80,-0.41,-1.42,1.62,-1.01,0.43,0.64,0.33,-0.43,-0.53,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,2.49,-0.99,0.00
355,0.80,-1.04,0.21,-0.07,-1.06,0.43,0.58,-0.47,-0.05,-1.30,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,1.97,-0.40,-0.99,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23,0.80,1.42,-0.16,1.14,0.91,-1.70,0.30,-0.95,0.01,-0.53,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,2.49,-0.99,0.00
638,0.39,-0.46,0.21,-0.19,-0.45,0.43,0.64,0.56,0.61,-1.52,...,-0.09,0.00,-0.18,0.00,-0.21,7.33,-0.51,-0.40,-0.99,0.00
378,-0.51,-0.79,-0.29,0.35,-0.97,0.43,0.98,0.68,0.62,0.34,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00
301,0.39,0.69,1.34,0.29,0.50,0.43,-1.18,-0.50,-0.35,0.62,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00


,arr_clouds,arr_dewpt,arr_pres,arr_rh,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd,dep_clouds,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
476,-1.40,1.09,-0.04,0.35,0.87,0.43,0.13,-0.73,-0.59,1.02,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00
418,0.80,0.87,-1.92,1.26,0.33,-0.79,1.04,1.09,1.05,0.62,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00
667,0.80,-0.80,-1.04,1.62,-1.38,-3.53,1.10,1.06,0.37,0.62,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00
423,-0.80,0.83,0.84,-0.37,0.94,0.43,-1.18,-0.41,-0.48,0.62,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00
549,0.80,-0.28,0.96,-0.13,-0.29,0.43,0.87,0.86,0.25,-0.53,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,2.49,-0.99,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,0.39,0.07,-0.66,-2.78,2.05,0.43,1.21,-0.94,-0.41,0.62,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,-0.51,-0.40,1.01,0.00
628,0.39,-1.71,0.46,-1.39,-1.13,0.43,0.30,0.95,1.86,-1.30,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,1.97,-0.40,-0.99,0.00
518,0.39,1.36,-0.41,-0.19,1.39,0.43,-0.78,0.09,-1.23,-1.30,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,1.97,-0.40,-0.99,0.00
172,0.80,-1.11,1.72,-0.55,-0.94,0.43,0.75,-1.32,-0.59,-1.30,...,-0.09,0.00,-0.18,0.00,-0.21,-0.14,1.97,-0.40,-0.99,0.00


101    0
243    0
110    2
313    1
355    0
      ..
23     2
638    2
378    2
301    0
54     1
Name: arr_status_x, Length: 548, dtype: int32

476    0
418    0
667    0
423    2
549    0
      ..
104    2
628    2
518    1
172    0
2      0
Name: arr_status_x, Length: 137, dtype: int32

In [763]:
#model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'ovr', penalty = 'none', max_iter = 1000)
model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'multinomial', penalty = None, max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 10000)

# While using multiclass case do multi_class = 'ovr' or 'auto'; can also try other solvers
# While doing regularization, use penalty = 'l2' and also C = 10.0 (need to try other values too)

model.fit(X_train, y_train) 

# The following gives the mean accuracy on the given data and labels
model.score(X_train, y_train) 

# This is the coefficient Beta_1, ..., Beta_7
model.coef_

# This is the coefficient Beta_0
model.intercept_

LogisticRegression(max_iter=1000, multi_class='multinomial', penalty=None)

0.677007299270073

array([[-8.01681045e-01,  1.04341773e+00, -3.80978381e-01,
        -5.59548760e-01, -1.31641894e+00, -4.31323178e-02,
        -7.80971332e-02,  1.11635137e-01, -1.78379973e-01,
        -1.49668262e-01, -8.88329315e-01,  4.94606985e-01,
         4.59746620e-01,  1.05957124e+00,  2.48677795e-01,
        -9.85397491e-02, -3.20136796e-01,  1.60108768e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  4.13236784e-01,
         0.00000000e+00,  6.94824176e-02,  0.00000000e+00,
        -7.14677384e-02, -6.62479945e-01,  1.08469437e+00,
         0.00000000e+00,  0.00000000e+00, -1.56198907e+00,
         1.84207153e+00,  0.00000000e+00,  0.00000000e+00,
        -2.73425113e-01, -7.24664084e-01,  1.30333097e-01,
         2.20624742e-02,  1.98906031e-02,  6.70587530e-02,
         5.80575883e-02,  1.09960365e-01,  9.98821451e-01,
        -8.47578369e-02,  0.00000000e+00,  0.00000000e+0

array([ 0.37657222, -0.15817322, -0.218399  ])

In [764]:
model.score(X_test,y_test)

0.44525547445255476

In [765]:
from sklearn.tree import DecisionTreeClassifier
# from sklearn.tree import DecisionTreeRegressor
clf = DecisionTreeClassifier(random_state=50, min_samples_leaf = 3)

clf = clf.fit(X_train, y_train) 
clf.score(X_train, y_train)

clf.feature_importances_
test_output = pd.DataFrame(clf.predict(X_test), index = X_test.index, columns = ['pred_arr_status'])
test_output.head()
clf.score(X_test, y_test)

0.8740875912408759

array([0.01866953, 0.04225537, 0.0419851 , 0.08603533, 0.02639326,
       0.00999799, 0.02632079, 0.04038087, 0.05047417, 0.01977827,
       0.10190679, 0.1333801 , 0.05342995, 0.03947341, 0.01444711,
       0.06307774, 0.08379938, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.00435622, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.01184626,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.02024055, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.00193078, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.01448725, 0.     

,pred_arr_status
476,2
418,0
667,0
423,0
549,1


0.31386861313868614

In [766]:

from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=50, min_samples_split = 12, min_samples_leaf = 6, max_depth = 4, n_estimators = 300)

gb = gb.fit(X_train, y_train) 
gb.score(X_train, y_train) 

1.0

In [767]:
gb.score(X_test,y_test)

0.40145985401459855

In [768]:
pred_data1 = pd.read_csv('pred_data2.csv')
pred_data1.head()
pred_data1.dtypes

,Unnamed: 0,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,arr_clouds,arr_dewpt,...,dep_clouds,dep_dewpt,dep_pres,dep_rh,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,1,19,2,ORD,22,2,59,52,76,6.60,...,89,3.60,982.50,45,15.50,24.13,804,70,4.20,3.20
1,3,14,2,JFK,16,2,55,21,80,8.80,...,91,8.30,"1,014.50",76,12.40,0.20,804,90,4.10,3.60
2,5,13,2,MCO,16,2,35,25,80,8.80,...,57,16.90,"1,014.50",52,27.60,31.70,803,150,9.66,6.40
3,7,19,3,ORD,22,3,59,52,89,11.50,...,81,5.20,971.50,70,10.40,24.13,804,320,17.20,11.60
4,9,14,3,JFK,16,3,55,21,87,11.00,...,94,9.80,"1,010.00",80,13.20,24.13,804,150,10.80,7.20


Unnamed: 0             int64
dep_hour               int64
dep_day                int64
Origin_Airport        object
arr_hour               int64
arr_day                int64
dep_min                int64
arr_min                int64
arr_clouds             int64
arr_dewpt            float64
arr_pres             float64
arr_rh                 int64
arr_temp             float64
arr_vis              float64
arr_weather.code       int64
arr_wind_dir           int64
arr_wind_gust_spd    float64
arr_wind_spd         float64
dep_clouds             int64
dep_dewpt            float64
dep_pres             float64
dep_rh                 int64
dep_temp             float64
dep_vis              float64
dep_weather.code       int64
dep_wind_dir           int64
dep_wind_gust_spd    float64
dep_wind_spd         float64
dtype: object

In [769]:

pred_data1['dep_min'] = pred_data1['dep_min'].astype('object')
pred_data1['arr_min'] = pred_data1['arr_min'].astype('object')
pred_data1['dep_hour'] = pred_data1['dep_hour'].astype('object')
pred_data1['dep_day'] = pred_data1['dep_day'].astype('object')
pred_data1['arr_hour'] = pred_data1['arr_hour'].astype('object')
pred_data1['arr_day'] = pred_data1['arr_day'].astype('object')
pred_data1['dep_weather.code'] = pred_data1['dep_weather.code'].astype('object')
pred_data1['arr_weather.code'] = pred_data1['arr_weather.code'].astype('object')
pred_data1.drop(columns=['Unnamed: 0'],inplace=True)

pred_data1.dtypes

dep_hour              object
dep_day               object
Origin_Airport        object
arr_hour              object
arr_day               object
dep_min               object
arr_min               object
arr_clouds             int64
arr_dewpt            float64
arr_pres             float64
arr_rh                 int64
arr_temp             float64
arr_vis              float64
arr_weather.code      object
arr_wind_dir           int64
arr_wind_gust_spd    float64
arr_wind_spd         float64
dep_clouds             int64
dep_dewpt            float64
dep_pres             float64
dep_rh                 int64
dep_temp             float64
dep_vis              float64
dep_weather.code      object
dep_wind_dir           int64
dep_wind_gust_spd    float64
dep_wind_spd         float64
dtype: object

In [770]:
pred_data1['dep_hour'] = pd.Categorical(pred_data1['dep_hour'], categories=[i for i in range(24)])
pred_data1['dep_day'] = pd.Categorical(pred_data1['dep_day'], categories=[i for i in range(7)])
pred_data1['dep_min'] = pd.Categorical(pred_data1['dep_min'], categories=[i for i in range(60)])
pred_data1['arr_hour'] = pd.Categorical(pred_data1['arr_hour'], categories=[i for i in range(24)])
pred_data1['arr_day'] = pd.Categorical(pred_data1['arr_day'], categories=[i for i in range(7)])
pred_data1['arr_min'] = pd.Categorical(pred_data1['arr_min'], categories=[i for i in range(60)])
#su_data['Carrier_Code'] = pd.Categorical(su_data['Carrier_Code'], categories=['AA', 'UA', 'DL', 'B6', 'WN'])
pred_data1['Origin_Airport'] = pd.Categorical(pred_data1['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])

pred_data1['arr_weather.code'] = pd.Categorical(pred_data1['arr_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
pred_data1['dep_weather.code'] = pd.Categorical(pred_data1['dep_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])



In [771]:
len(pred_data1)

12

In [772]:
my_data = {'arr_status_y': [0, 1, 2]}
df = pd.DataFrame(data = my_data)

In [773]:
df.head()

,arr_status_y
0,0
1,1
2,2


In [774]:
pred_data1 =pred_data1.merge(df,how = "cross")
len(pred_data1)

36

In [775]:
pred_data1['arr_status_y'] = pd.Categorical(pred_data1['arr_status_y'], categories=[0,1,2])

In [776]:
pred_data1 = pred_data1[['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'arr_status_y', 'arr_clouds',
       'arr_dewpt', 'arr_pres', 'arr_rh', 'arr_temp', 'arr_vis',
       'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd', 'arr_wind_spd',
       'dep_clouds', 'dep_dewpt', 'dep_pres', 'dep_rh', 'dep_temp', 'dep_vis',
       'dep_weather.code', 'dep_wind_dir', 'dep_wind_gust_spd',
       'dep_wind_spd']]

In [777]:
pred_data1 = pd.get_dummies(pred_data1, drop_first = True)
#su_data['arr_status'] = su_data['arr_status'].astype('object')
pred_data1.head()
pred_data1.dtypes
pred_data1.columns

,arr_clouds,arr_dewpt,arr_pres,arr_rh,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd,dep_clouds,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
0,76,6.60,999.00,67,12.60,24.13,110,4.06,2.53,89,...,False,False,False,False,False,False,False,False,True,False
1,76,6.60,999.00,67,12.60,24.13,110,4.06,2.53,89,...,False,False,False,False,False,False,False,False,True,False
2,76,6.60,999.00,67,12.60,24.13,110,4.06,2.53,89,...,False,False,False,False,False,False,False,False,True,False
3,80,8.80,998.00,58,17.10,24.13,286,6.10,4.13,91,...,False,False,False,False,False,False,False,False,True,False
4,80,8.80,998.00,58,17.10,24.13,286,6.10,4.13,91,...,False,False,False,False,False,False,False,False,True,False


arr_clouds                int64
arr_dewpt               float64
arr_pres                float64
arr_rh                    int64
arr_temp                float64
                         ...   
dep_weather.code_801       bool
dep_weather.code_802       bool
dep_weather.code_803       bool
dep_weather.code_804       bool
dep_weather.code_900       bool
Length: 268, dtype: object

Index(['arr_clouds', 'arr_dewpt', 'arr_pres', 'arr_rh', 'arr_temp', 'arr_vis',
       'arr_wind_dir', 'arr_wind_gust_spd', 'arr_wind_spd', 'dep_clouds',
       ...
       'dep_weather.code_721', 'dep_weather.code_731', 'dep_weather.code_741',
       'dep_weather.code_751', 'dep_weather.code_800', 'dep_weather.code_801',
       'dep_weather.code_802', 'dep_weather.code_803', 'dep_weather.code_804',
       'dep_weather.code_900'],
      dtype='object', length=268)

In [778]:
from sklearn.preprocessing import StandardScaler
#sc = StandardScaler()
X_test = pd.DataFrame(sc.transform(pred_data1), columns = pred_data1.columns, index = pred_data1.index)


In [780]:
linear_reg_output = pd.DataFrame(gb.predict(X_test), index = X_test.index, columns = ['pred_arr_status'])
#linear_reg_output = linear_reg_output.merge(y_test, left_index = True, right_index = True)
linear_reg_output.head(50)
# mean_absolute_error = abs(linear_reg_output['pred_delay'] - linear_reg_output['arr_delay']).mean()
# print('Mean absolute error is ')
# print(mean_absolute_error)
# maer = mean_absolute_error/linear_reg_output['pred_delay'].mean()
# print('Mean absolute error ratio is ')
# print(maer)
#model.score(X_test, y_test)

,pred_arr_status
0,1
1,1
2,1
3,0
4,1
5,1
6,0
7,0
8,0
9,2
